# NB05 — Wildfire Risk Prediction & Interactive Dashboard

**Goal:** Use the weather forecasts from NB03 + the fire models from NB04 to predict **future wildfire risk** at both daily (30-day) and hourly (168h) granularity, then build interactive maps for a jury demo.

## Part A — Daily Risk (30-day horizon)
`§1` Load forecast + fire model → `§2` Engineer features → `§3` Predict risk → `§4` Folium maps → `§5` Plotly dashboard → `§6` Dashboard JSON/CSV export

## Part B — Hourly Risk (168-hour horizon)
Load hourly forecast + hourly fire model → engineer features → predict → JSON export

**Input:** `outputs/weather_forecast_30d.parquet`, `outputs/weather_forecast_168h.parquet`, `models/wildfire/best_fire_model*.joblib`  
**Output:** `outputs/wildfire_risk_30d.parquet`, `outputs/wildfire_risk_168h.parquet`, `outputs/hourly_forecast_168h.json`

In [1]:
# ─── Cell 1: Imports ─────────────────────────────────────────────────────
import os, sys, json, warnings
from pathlib import Path

for _p in ["folium", "plotly"]:
    try: __import__(_p)
    except ImportError:
        import subprocess; subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _p])

import numpy as np, pandas as pd
from joblib import load as jl_load
import folium
import plotly.express as px

warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from src.config import (ROOT, PROCESSED, OUTPUTS, MODELS_F, REFERENCE,
                         CITIES, FORECAST_30D, RISK_30D, ENG_DAILY, MASTER_DAILY)

MAPS    = ROOT / "reports" / "maps"
FIGURES = ROOT / "reports" / "figures"
for p in (MAPS, FIGURES): p.mkdir(parents=True, exist_ok=True)
print(f"Root: {ROOT}")

Root: /home/manheim666/Desktop/WildFire-Prediction


In [2]:
# ─── §1: Load forecast, model, historical data ──────────────────────────

# 1a. Weather forecast
assert FORECAST_30D.exists(), f"Run NB03 first — {FORECAST_30D} not found"
fc = pd.read_parquet(FORECAST_30D)
fc["Date"] = pd.to_datetime(fc["Date"])
print(f"Forecast: {fc.shape}  {fc['Date'].min().date()} → {fc['Date'].max().date()}")

# Ensure ALL cities are represented (broadcast if missing)
if "City" not in fc.columns:
    # No City column → broadcast to all cities
    rows = [fc.assign(City=c) for c in CITIES]
    fc = pd.concat(rows, ignore_index=True)
    print(f"  Broadcast to {len(CITIES)} cities → {fc.shape}")
else:
    # City column exists — ensure ALL 16 cities present
    present = set(fc["City"].unique())
    missing = set(CITIES.keys()) - present
    if missing:
        # For missing cities, duplicate the template row (same dates/weather)
        template = fc[fc["City"] == list(present)[0]].copy()
        extras = []
        for city in missing:
            t = template.copy()
            t["City"] = city
            extras.append(t)
        fc = pd.concat([fc] + extras, ignore_index=True)
        print(f"  Added {len(missing)} missing cities: {sorted(missing)}")

fc_cities = sorted(fc["City"].unique())
print(f"  Cities in forecast ({len(fc_cities)}): {fc_cities}")

# 1b. Fire model + manifest
assert (MODELS_F / "best_fire_model.joblib").exists(), "Run NB04 first"
fire_model = jl_load(MODELS_F / "best_fire_model.joblib")

with open(MODELS_F / "model_manifest.json") as f:
    manifest = json.load(f)
THRESHOLD = manifest["optimal_threshold"]
print(f"Model: {manifest['model_name']}  threshold={THRESHOLD:.4f}")

with open(MODELS_F / "feature_columns.json") as f:
    FEATURE_COLS = json.load(f)
print(f"Features: {len(FEATURE_COLS)}")

# 1c. Historical data for lag features
hist_path = ENG_DAILY if ENG_DAILY.exists() else MASTER_DAILY
hist = pd.read_parquet(hist_path)
hist["Date"] = pd.to_datetime(hist["Date"])
hist = hist.sort_values(["City", "Date"])
print(f"History: {hist.shape}")

# 1d. Static geography
static_path = REFERENCE / "static_geography.parquet"
static_df = (pd.read_parquet(static_path) if static_path.exists()
             else pd.DataFrame([{"City": c, "Latitude": lat, "Longitude": lon,
                                  "Elevation": 0, "Slope": 0, "Trees_pct": 0,
                                  "Urban_pct": 0, "Pop_Total": 0}
                                 for c, (lat, lon) in CITIES.items()]))

Forecast: (480, 10)  2026-05-04 → 2026-06-02
  Cities in forecast (16): ['Baku', 'Barda', 'Gabala', 'Ganja', 'Jalilabad', 'Khachmaz', 'Lankaran', 'Mingachevir', 'Nakhchivan', 'Quba', 'Shabran', 'Shaki', 'Shamakhi', 'Shirvan', 'Yevlakh', 'Zaqatala']
Model: CatBoost_Optuna  threshold=0.5000
Features: 184
History: (83503, 269)


In [3]:
# ─── §2: Engineer features for forecast period ──────────────────────────

# Expand forecast to all cities if needed
if "City" not in fc.columns:
    rows = [fc.assign(City=c) for c in CITIES]
    fc = pd.concat(rows, ignore_index=True)
    print(f"Broadcast to {len(CITIES)} cities → {fc.shape}")

# Rename bare weather columns → _mean suffix (match training feature names)
rename_map = {}
for c in fc.columns:
    if c in ("City", "Date"): continue
    if not any(s in c for s in ("_mean","_sum","_min","_max")):
        if c + "_mean" not in fc.columns:
            rename_map[c] = c + "_mean"
if rename_map:
    fc = fc.rename(columns=rename_map)

# Build feature matrix per city (history concat for lags)
frames = []
for city in sorted(fc["City"].unique()):
    cf = fc[fc["City"] == city].sort_values("Date").copy()
    ch = hist[hist["City"] == city].sort_values("Date")
    cutoff = cf["Date"].min() - pd.Timedelta(days=90)
    recent = ch[ch["Date"] >= cutoff]

    common = ["City","Date"] + [c for c in cf.columns
              if c in recent.columns and c not in ("City","Date")]
    combo = pd.concat([recent[common], cf[common]], ignore_index=True)
    combo = combo.sort_values("Date").drop_duplicates(["City","Date"], keep="last")

    # Calendar
    combo["Year"]       = combo["Date"].dt.year
    combo["Month"]      = combo["Date"].dt.month
    combo["DayOfYear"]  = combo["Date"].dt.dayofyear
    combo["DayOfWeek"]  = combo["Date"].dt.dayofweek
    combo["WeekOfYear"] = combo["Date"].dt.isocalendar().week.astype(int)
    for col, period in [("Month",12),("DayOfYear",365),("DayOfWeek",7)]:
        combo[f"{col}_sin"] = np.sin(2*np.pi*combo[col]/period)
        combo[f"{col}_cos"] = np.cos(2*np.pi*combo[col]/period)
    combo["Month_sin"]  = np.sin(2*np.pi*combo["Month"]/12)
    combo["Month_cos"]  = np.cos(2*np.pi*combo["Month"]/12)
    combo["DoY_sin"]    = np.sin(2*np.pi*combo["DayOfYear"]/365)
    combo["DoY_cos"]    = np.cos(2*np.pi*combo["DayOfYear"]/365)
    combo["DoW_sin"]    = np.sin(2*np.pi*combo["DayOfWeek"]/7)
    combo["DoW_cos"]    = np.cos(2*np.pi*combo["DayOfWeek"]/7)
    combo["Season"]     = combo["Month"].map(
        {12:0,1:0,2:0,3:1,4:1,5:1,6:2,7:2,8:2,9:3,10:3,11:3})
    combo["is_summer"]      = combo["Month"].isin([6,7,8]).astype(int)
    combo["is_fire_season"] = combo["Month"].isin([5,6,7,8,9]).astype(int)

    # Lag / rolling from numeric cols
    num_cols = [c for c in combo.columns if combo[c].dtype in
                ("float64","float32","int64") and c not in
                ("City","Date","Year","Month","DayOfYear","DayOfWeek","WeekOfYear")]
    for v in num_cols[:15]:  # cap to top-15 to avoid explosion
        for lag in [1,2,3,7,14,30]:
            combo[f"{v}_lag{lag}"] = combo[v].shift(lag)
        for w in [3,7,14,30]:
            combo[f"{v}_roll{w}_mean"] = combo[v].shift(1).rolling(w,min_periods=1).mean()

    combo = combo.merge(static_df, on="City", how="left")
    frames.append(combo[combo["Date"].isin(cf["Date"].values)])

forecast_df = pd.concat(frames, ignore_index=True).fillna(0)
print(f"Feature matrix: {forecast_df.shape}")

Feature matrix: (480, 187)


In [4]:
# ─── §3: Predict risk ────────────────────────────────────────────────────

# Align columns
for col in FEATURE_COLS:
    if col not in forecast_df.columns:
        forecast_df[col] = 0
X_fc = forecast_df[FEATURE_COLS].fillna(0)
print(f"Prediction matrix: {X_fc.shape}")

fire_proba = fire_model.predict_proba(X_fc)[:, 1]

risk_df = forecast_df[["City", "Date"]].copy()
for col in ["Temperature_C_mean","Humidity_percent_mean","Rain_mm_sum",
            "Wind_Speed_kmh_mean","Pressure_hPa_mean"]:
    if col in forecast_df.columns:
        risk_df[col] = forecast_df[col].values

risk_df["fire_probability"] = fire_proba
risk_df["fire_predicted"]   = (fire_proba >= THRESHOLD).astype(int)
risk_df["risk_level"] = pd.cut(
    fire_proba, bins=[-1, 0.15, 0.35, 0.60, 1.01],
    labels=["Low", "Medium", "High", "Extreme"])

coords = pd.DataFrame([{"City":c,"Latitude":lat,"Longitude":lon}
                        for c,(lat,lon) in CITIES.items()])
risk_df = risk_df.merge(coords, on="City", how="left")

risk_df.to_parquet(RISK_30D, index=False)
risk_df.to_csv(OUTPUTS / "wildfire_risk_30d.csv", index=False)

print(f"\nRisk saved: {risk_df.shape}  → {RISK_30D.name}")
print(f"\nRisk distribution:\n{risk_df['risk_level'].value_counts().to_string()}")
print(f"\nTop 10 highest-risk:")
print(risk_df.nlargest(10, "fire_probability")[
    ["City","Date","fire_probability","risk_level"]].to_string(index=False))
print(f"\nMean risk by city:")
print(risk_df.groupby("City")["fire_probability"].agg(["mean","max"])
      .sort_values("mean", ascending=False).round(4).to_string())

Prediction matrix: (480, 184)

Risk saved: (480, 12)  → wildfire_risk_30d.parquet

Risk distribution:
risk_level
Low        480
Medium       0
High         0
Extreme      0

Top 10 highest-risk:
     City       Date  fire_probability risk_level
     Baku 2026-05-25          0.025002        Low
     Baku 2026-06-02          0.024282        Low
Jalilabad 2026-06-02          0.023828        Low
     Baku 2026-05-20          0.023601        Low
 Lankaran 2026-05-26          0.022405        Low
     Baku 2026-05-29          0.022291        Low
Jalilabad 2026-05-26          0.022242        Low
Jalilabad 2026-05-11          0.022061        Low
    Barda 2026-05-19          0.021357        Low
 Lankaran 2026-05-27          0.021237        Low

Mean risk by city:
               mean     max
City                       
Baku         0.0185  0.0250
Barda        0.0153  0.0214
Jalilabad    0.0151  0.0238
Ganja        0.0148  0.0202
Mingachevir  0.0147  0.0208
Lankaran     0.0137  0.0224
Shirvan    

## §4 — Folium Maps (Per-Date)

Colour-coded circle markers on Azerbaijan. Click for details.

In [5]:
# ─── §4: Folium maps ─────────────────────────────────────────────────────
RISK_COLORS = {"Low":"green","Medium":"orange","High":"red","Extreme":"darkred"}
AZ_CENTER, AZ_ZOOM = [40.4, 47.8], 7

def _folium_map(day_df, date_str):
    m = folium.Map(location=AZ_CENTER, zoom_start=AZ_ZOOM, tiles="CartoDB positron")
    title = (f'<div style="position:fixed;top:10px;left:60px;z-index:1000;'
             f'background:white;padding:10px 20px;border-radius:8px;'
             f'box-shadow:0 2px 6px rgba(0,0,0,.3);font-family:Arial">'
             f'<h3 style="margin:0">Azerbaijan Wildfire Risk — {date_str}</h3></div>')
    m.get_root().html.add_child(folium.Element(title))
    for _, r in day_df.iterrows():
        prob = r.get("fire_probability", 0)
        lvl  = str(r.get("risk_level", "Low"))
        col  = RISK_COLORS.get(lvl, "gray")
        popup = [f"<b>{r['City']}</b><br>Date: {date_str}<br>"
                 f"Fire Risk: {prob*100:.1f}% (<b style='color:{col}'>{lvl}</b>)<br>"]
        for c, lbl, u in [("Temperature_C_mean","Temp","°C"),
                           ("Humidity_percent_mean","Hum","%"),
                           ("Wind_Speed_kmh_mean","Wind"," km/h"),
                           ("Rain_mm_sum","Rain"," mm")]:
            if c in r.index and pd.notna(r[c]):
                popup.append(f"{lbl}: {r[c]:.1f}{u}<br>")
        folium.CircleMarker([r["Latitude"], r["Longitude"]],
            radius=max(8, prob*40), color=col, fill=True,
            fill_color=col, fill_opacity=0.7,
            popup=folium.Popup("".join(popup), max_width=250),
            tooltip=f"{r['City']}: {prob*100:.0f}%").add_to(m)
    legend = ('<div style="position:fixed;bottom:30px;left:30px;z-index:1000;'
              'background:white;padding:10px 15px;border-radius:8px;'
              'box-shadow:0 2px 6px rgba(0,0,0,.3);font-size:12px">'
              '<b>Risk</b><br>'
              '<span style="color:green">●</span> Low<br>'
              '<span style="color:orange">●</span> Medium<br>'
              '<span style="color:red">●</span> High<br>'
              '<span style="color:darkred">●</span> Extreme</div>')
    m.get_root().html.add_child(folium.Element(legend))
    return m

dates = sorted(risk_df["Date"].unique())
key_dates = sorted(set([dates[0]] + list(dates[4::5]) + [dates[-1]]))

for dt in key_dates:
    ds = pd.Timestamp(dt).strftime("%Y-%m-%d")
    dd = risk_df[risk_df["Date"] == dt]
    if dd.empty: continue
    _folium_map(dd, ds).save(str(MAPS / f"fire_risk_{ds}.html"))
    print(f"  ✓ {ds}")

demo = _folium_map(risk_df[risk_df["Date"] == dates[0]],
                   pd.Timestamp(dates[0]).strftime("%Y-%m-%d"))
demo.save(str(MAPS / "fire_risk_demo.html"))
print(f"Demo map: {MAPS / 'fire_risk_demo.html'}")

  ✓ 2026-05-04
  ✓ 2026-05-08
  ✓ 2026-05-13
  ✓ 2026-05-18
  ✓ 2026-05-23
  ✓ 2026-05-28
  ✓ 2026-06-02
Demo map: /home/manheim666/Desktop/WildFire-Prediction/reports/maps/fire_risk_demo.html


## §5 — Plotly Animated Dashboard + Weather Maps

In [6]:
# ─── §5: Plotly dashboard ────────────────────────────────────────────────
plot_df = risk_df.copy()
plot_df["date_str"] = plot_df["Date"].dt.strftime("%Y-%m-%d")
plot_df["fire_pct"] = (plot_df["fire_probability"] * 100).round(1)

fig = px.scatter_geo(
    plot_df, lat="Latitude", lon="Longitude",
    color="fire_pct", size="fire_pct",
    hover_name="City",
    hover_data={"fire_pct":True, "risk_level":True,
                "Latitude":False, "Longitude":False, "date_str":False},
    animation_frame="date_str",
    color_continuous_scale=["green","yellow","orange","red","darkred"],
    range_color=[0, 80], size_max=30,
    title="Azerbaijan Wildfire Risk Forecast — 30-Day Outlook",
)
fig.update_geos(center=dict(lat=40.3,lon=47.8), projection_scale=15,
    showland=True, landcolor="rgb(243,243,243)",
    showocean=True, oceancolor="rgb(204,229,255)",
    showcountries=True)
fig.update_layout(height=700, width=1000,
    coloraxis_colorbar=dict(title="Risk %"))
fig.write_html(str(MAPS / "fire_risk_dashboard.html"), include_plotlyjs="cdn")
print(f"Dashboard: {MAPS / 'fire_risk_dashboard.html'}")
fig.show()

# ── Weather variable maps ────────────────────────────────────────────────
for var, label, cmap in [
    ("Temperature_C_mean",    "Temperature (°C)", "RdYlBu_r"),
    ("Humidity_percent_mean", "Humidity (%)",      "YlGnBu"),
    ("Wind_Speed_kmh_mean",   "Wind (km/h)",       "Purples"),
    ("Rain_mm_sum",           "Rain (mm)",          "Blues"),
]:
    if var not in plot_df.columns: continue
    sub = plot_df.dropna(subset=[var])
    if sub.empty: continue
    fw = px.scatter_geo(
        sub, lat="Latitude", lon="Longitude",
        color=var, size=np.abs(sub[var])+1,
        hover_name="City", animation_frame="date_str",
        color_continuous_scale=cmap, size_max=25,
        title=f"Azerbaijan {label} — 30-Day Forecast",
    )
    fw.update_geos(center=dict(lat=40.3,lon=47.8), projection_scale=15,
        showland=True, landcolor="rgb(243,243,243)", showcountries=True)
    fw.update_layout(height=600, width=900)
    fname = f"weather_{var.split('_')[0].lower()}_dashboard.html"
    fw.write_html(str(MAPS / fname), include_plotlyjs="cdn")
    print(f"  ✓ {label} → {fname}")

print(f"\nAll maps in: {MAPS}")
for f in sorted(MAPS.glob("*.html")):
    print(f"  {f.name} ({f.stat().st_size/1024:.0f} KB)")
print(f"\n→ Next: 06_Climate_Report.ipynb")

Dashboard: /home/manheim666/Desktop/WildFire-Prediction/reports/maps/fire_risk_dashboard.html


  ✓ Temperature (°C) → weather_temperature_dashboard.html
  ✓ Humidity (%) → weather_humidity_dashboard.html
  ✓ Wind (km/h) → weather_wind_dashboard.html
  ✓ Rain (mm) → weather_rain_dashboard.html

All maps in: /home/manheim666/Desktop/WildFire-Prediction/reports/maps
  fire_risk_2026-04-30.html (26 KB)
  fire_risk_2026-05-01.html (24 KB)
  fire_risk_2026-05-02.html (26 KB)
  fire_risk_2026-05-03.html (26 KB)
  fire_risk_2026-05-04.html (26 KB)
  fire_risk_2026-05-05.html (24 KB)
  fire_risk_2026-05-06.html (26 KB)
  fire_risk_2026-05-07.html (26 KB)
  fire_risk_2026-05-08.html (26 KB)
  fire_risk_2026-05-09.html (26 KB)
  fire_risk_2026-05-10.html (24 KB)
  fire_risk_2026-05-11.html (26 KB)
  fire_risk_2026-05-12.html (26 KB)
  fire_risk_2026-05-13.html (26 KB)
  fire_risk_2026-05-14.html (26 KB)
  fire_risk_2026-05-15.html (24 KB)
  fire_risk_2026-05-16.html (26 KB)
  fire_risk_2026-05-17.html (26 KB)
  fire_risk_2026-05-18.html (26 KB)
  fire_risk_2026-05-19.html (26 KB)
  fire_ri

In [7]:
# ─── §6: Generate dashboard JSON/CSV — forecast_30_days ──────────────────
# Produces the same format that the web dashboard (dashboard/ & docs/) expects,
# then copies the files so the site is immediately up-to-date after a notebook run.
import shutil

RISK_COLORS_HEX = {
    "Low": "#3FA773", "Moderate": "#D8A31D",
    "High": "#D96C3B", "Extreme": "#B73333",
}

def _confidence(prob):
    """Readable confidence proxy: distance from the uncertain middle."""
    return float(np.clip(0.55 + abs(prob - 0.5) * 0.8, 0.55, 0.95))

def _risk_label(prob):
    if prob >= 0.60: return "Extreme"
    if prob >= 0.35: return "High"
    if prob >= 0.15: return "Moderate"
    return "Low"

def _climate_summary(row):
    temp = row.get("Temperature_C_mean", np.nan)
    wind = row.get("Wind_Speed_kmh_mean", np.nan)
    humidity = row.get("Humidity_percent_mean", np.nan)
    rain = row.get("Rain_mm_sum", np.nan)
    parts = []
    if pd.notna(temp) and temp >= 28:   parts.append("hot conditions")
    elif pd.notna(temp) and temp <= 12: parts.append("cool conditions")
    else:                                parts.append("mild temperatures")
    if pd.notna(wind) and wind >= 18:    parts.append("strong wind")
    if pd.notna(humidity) and humidity <= 40: parts.append("dry air")
    if pd.notna(rain) and rain >= 2:     parts.append("recent rainfall")
    return ", ".join(parts).capitalize() + "."

def _warning_text(row):
    if row["risk_level"] in ("High", "Extreme"):
        return "High temperature, dry air, and wind can accelerate wildfire spread."
    if row.get("Wind_Speed_kmh_mean", 0) >= 18:
        return "Wind is elevated, so small ignitions could spread faster."
    if row.get("Humidity_percent_mean", 100) <= 40:
        return "Low humidity can dry vegetation and raise ignition sensitivity."
    return "Current conditions suggest limited short-term wildfire pressure."

# ── Build dashboard dataframe ────────────────────────────────────────────
out = risk_df.copy()

# Pull extra weather cols from forecast_df
for col in ["Solar_Radiation_Wm2_mean", "Soil_Temp_C_mean", "Soil_Moisture_mean"]:
    if col in forecast_df.columns and col not in out.columns:
        lookup = forecast_df[["City", "Date", col]].drop_duplicates()
        out = out.merge(lookup, on=["City", "Date"], how="left")

# Dashboard-compatible columns
out["risk_level"]     = out["fire_probability"].map(_risk_label)
out["probability"]    = out["fire_probability"]
out["confidence"]     = out["fire_probability"].map(_confidence)
out["risk_score"]     = (out["fire_probability"] * 100).round(1)
out["predicted_fire"] = out["fire_predicted"]
out["temperature"]    = out["Temperature_C_mean"].round(1)
out["wind"]           = out["Wind_Speed_kmh_mean"].round(1)
out["humidity"]       = out["Humidity_percent_mean"].round(1)
out["rain"]           = out["Rain_mm_sum"].round(2)
out["climate_summary"] = out.apply(_climate_summary, axis=1)
out["warning"]        = out.apply(_warning_text, axis=1)
out["risk_color"]     = out["risk_level"].map(RISK_COLORS_HEX)
out["date"]           = out["Date"].dt.strftime("%Y-%m-%d")
out["region"]         = out["City"]

public_cols = [
    "date", "region", "risk_level", "probability", "confidence",
    "risk_score", "predicted_fire", "temperature", "wind", "humidity", "rain",
    "Temperature_C_mean", "Humidity_percent_mean", "Rain_mm_sum",
    "Wind_Speed_kmh_mean", "Pressure_hPa_mean",
    "Solar_Radiation_Wm2_mean", "Soil_Temp_C_mean", "Soil_Moisture_mean",
    "Latitude", "Longitude", "climate_summary", "warning", "risk_color",
]
public_cols = [c for c in public_cols if c in out.columns]
out_public = out[public_cols].copy()

# ── Save to outputs/ ─────────────────────────────────────────────────────
out_public.to_csv(OUTPUTS / "forecast_30_days.csv", index=False)
(OUTPUTS / "forecast_30_days.json").write_text(
    out_public.to_json(orient="records", indent=2), encoding="utf-8")

# ── metrics.json ─────────────────────────────────────────────────────────
metrics_out = {
    "generated_at": pd.Timestamp.now(tz="Asia/Baku").isoformat(),
    "prediction_horizon_days": 30,
    "target": "Daily probability of a NASA FIRMS wildfire detection within the city risk area",
    "selected_model": manifest["model_name"],
    "selected_threshold": float(THRESHOLD),
    "feature_count": len(FEATURE_COLS),
    "risk_levels": {
        "Low": "< 15%", "Moderate": "15% to 35%",
        "High": "35% to 60%", "Extreme": ">= 60%",
    },
    "data_sources": [
        "NASA FIRMS MODIS/VIIRS",
        "Open-Meteo ERA5/ERA5-Land",
        "Open-Elevation/static geography",
    ],
}
(OUTPUTS / "metrics.json").write_text(
    json.dumps(metrics_out, indent=2), encoding="utf-8")

# ── Copy to dashboard & docs ─────────────────────────────────────────────
for dest in [ROOT / "dashboard" / "data", ROOT / "docs" / "data"]:
    dest.mkdir(parents=True, exist_ok=True)
    for fname in ["forecast_30_days.csv", "forecast_30_days.json", "metrics.json"]:
        shutil.copy2(OUTPUTS / fname, dest / fname)

print(f"Dashboard outputs saved ({len(out_public)} rows):")
print(f"  Date range: {out_public['date'].min()} → {out_public['date'].max()}")
print(f"  Regions:    {sorted(out_public['region'].unique())}")
print(f"  Copied to:  dashboard/data/ + docs/data/")
print(f"\n→ Next: 06_Climate_Report.ipynb")

Dashboard outputs saved (480 rows):
  Date range: 2026-05-04 → 2026-06-02
  Regions:    ['Baku', 'Barda', 'Gabala', 'Ganja', 'Jalilabad', 'Khachmaz', 'Lankaran', 'Mingachevir', 'Nakhchivan', 'Quba', 'Shabran', 'Shaki', 'Shamakhi', 'Shirvan', 'Yevlakh', 'Zaqatala']
  Copied to:  dashboard/data/ + docs/data/

→ Next: 06_Climate_Report.ipynb


---

# PART B — Hourly Fire Risk Prediction (168h)

Uses the hourly fire model from NB04 + 168h weather forecast from NB03 to produce **hour-level wildfire risk** for every city × hour.

**Input:** `outputs/weather_forecast_168h.parquet`, `models/wildfire/best_fire_model_hourly.joblib`  
**Output:** `outputs/wildfire_risk_168h.parquet`, `outputs/hourly_forecast_168h.json`

In [8]:
# ─── PART B: Hourly fire risk prediction ─────────────────────────────────
from src.config import FORECAST_168H, ENG_HOURLY, RISK_168H

print("=" * 60)
print("  HOURLY FIRE RISK PREDICTION (168h)")
print("=" * 60)

# B1. Load hourly weather forecast
assert FORECAST_168H.exists(), f"Run NB03 first — {FORECAST_168H} not found"
h_fc = pd.read_parquet(FORECAST_168H)
h_fc["Timestamp"] = pd.to_datetime(h_fc["Timestamp"])
if "Date" not in h_fc.columns:
    h_fc["Date"] = h_fc["Timestamp"].dt.normalize()
print(f"Hourly forecast: {h_fc.shape}  "
      f"{h_fc['Timestamp'].min()} → {h_fc['Timestamp'].max()}")

# B2. Load hourly fire model + manifest
h_model_path    = MODELS_F / "best_fire_model_hourly.joblib"
h_manifest_path = MODELS_F / "model_manifest_hourly.json"
h_feat_path     = MODELS_F / "feature_columns_hourly.json"

assert h_model_path.exists(), "Run NB04 first — hourly model not found"
h_fire_model = jl_load(h_model_path)

with open(h_manifest_path) as f:
    h_manifest = json.load(f)
H_THRESHOLD = h_manifest["optimal_threshold"]
print(f"Hourly model: {h_manifest['model_name']}  threshold={H_THRESHOLD:.4f}")

with open(h_feat_path) as f:
    H_FEATURE_COLS = json.load(f)
print(f"Hourly features: {len(H_FEATURE_COLS)}")

# B3. Load hourly history for lag features
h_hist = pd.read_parquet(ENG_HOURLY)
h_hist["Timestamp"] = pd.to_datetime(h_hist["Timestamp"])
h_hist["Date"] = pd.to_datetime(h_hist["Date"])
h_hist = h_hist.sort_values(["City", "Timestamp"])
print(f"Hourly history: {h_hist.shape}")

# B4. Engineer features for hourly forecast
h_frames = []
for city in sorted(h_fc["City"].unique()):
    cf = h_fc[h_fc["City"] == city].sort_values("Timestamp").copy()
    ch = h_hist[h_hist["City"] == city].sort_values("Timestamp")
    cutoff = cf["Timestamp"].min() - pd.Timedelta(hours=168*2)
    recent = ch[ch["Timestamp"] >= cutoff]

    common = ["City", "Date", "Timestamp"] + [
        c for c in cf.columns if c in recent.columns and c not in ("City", "Date", "Timestamp")]
    combo = pd.concat([recent[common], cf[common]], ignore_index=True)
    combo = combo.sort_values("Timestamp").drop_duplicates(["City", "Timestamp"], keep="last")

    # Calendar features
    combo["Year"]       = combo["Timestamp"].dt.year
    combo["Month"]      = combo["Timestamp"].dt.month
    combo["DayOfYear"]  = combo["Timestamp"].dt.dayofyear
    combo["DayOfWeek"]  = combo["Timestamp"].dt.dayofweek
    combo["Hour"]       = combo["Timestamp"].dt.hour
    combo["WeekOfYear"] = combo["Timestamp"].dt.isocalendar().week.astype(int)
    for col, period in [("Month",12),("DayOfYear",365),("DayOfWeek",7),("Hour",24)]:
        combo[f"{col}_sin"] = np.sin(2*np.pi*combo[col]/period)
        combo[f"{col}_cos"] = np.cos(2*np.pi*combo[col]/period)
    combo["Season"]         = combo["Month"].map(
        {12:0,1:0,2:0,3:1,4:1,5:1,6:2,7:2,8:2,9:3,10:3,11:3})
    combo["is_summer"]      = combo["Month"].isin([6,7,8]).astype(int)
    combo["is_fire_season"] = combo["Month"].isin([5,6,7,8,9]).astype(int)
    combo["is_daytime"]     = combo["Hour"].between(6, 20).astype(int)

    # Hourly lag / rolling
    num_cols = [c for c in combo.columns if combo[c].dtype in
                ("float64","float32","int64") and c not in
                ("City","Date","Timestamp","Year","Month","DayOfYear","DayOfWeek","Hour","WeekOfYear")]
    for v in num_cols[:10]:
        for lag in [1, 3, 6, 12, 24]:
            combo[f"{v}_lag{lag}"] = combo[v].shift(lag)
        for w in [6, 12, 24]:
            combo[f"{v}_roll{w}_mean"] = combo[v].shift(1).rolling(w, min_periods=1).mean()

    combo = combo.merge(static_df, on="City", how="left")
    h_frames.append(combo[combo["Timestamp"].isin(cf["Timestamp"].values)])

h_forecast_df = pd.concat(h_frames, ignore_index=True).fillna(0)
print(f"Hourly feature matrix: {h_forecast_df.shape}")

# B5. Predict hourly fire risk
for col in H_FEATURE_COLS:
    if col not in h_forecast_df.columns:
        h_forecast_df[col] = 0
X_hfc = h_forecast_df[H_FEATURE_COLS].fillna(0)
print(f"Hourly prediction matrix: {X_hfc.shape}")

h_fire_proba = h_fire_model.predict_proba(X_hfc)[:, 1]

h_risk = h_forecast_df[["City", "Timestamp", "Date"]].copy()
for col in ["Temperature_C", "Humidity_percent", "Wind_Speed_kmh", "Solar_Radiation_Wm2"]:
    if col in h_forecast_df.columns:
        h_risk[col] = h_forecast_df[col].values
h_risk["fire_probability"] = h_fire_proba
h_risk["fire_predicted"]   = (h_fire_proba >= H_THRESHOLD).astype(int)
h_risk["risk_level"] = pd.cut(
    h_fire_proba, bins=[-1, 0.15, 0.35, 0.60, 1.01],
    labels=["Low", "Medium", "High", "Extreme"])

coords = pd.DataFrame([{"City":c,"Latitude":lat,"Longitude":lon}
                        for c,(lat,lon) in CITIES.items()])
h_risk = h_risk.merge(coords, on="City", how="left")

h_risk.to_parquet(RISK_168H, index=False)
print(f"\nHourly risk saved: {h_risk.shape}  → {RISK_168H.name}")
print(f"Risk distribution:\n{h_risk['risk_level'].value_counts().to_string()}")
print(f"\nTop 10 highest hourly risk:")
print(h_risk.nlargest(10, "fire_probability")[
    ["City","Timestamp","fire_probability","risk_level"]].to_string(index=False))

  HOURLY FIRE RISK PREDICTION (168h)
Hourly forecast: (2688, 7)  2026-05-03 20:00:00 → 2026-05-10 19:00:00
Hourly model: CatBoost_H  threshold=0.5000
Hourly features: 68
Hourly history: (2003712, 99)
Hourly feature matrix: (2688, 114)
Hourly prediction matrix: (2688, 68)

Hourly risk saved: (2688, 12)  → wildfire_risk_168h.parquet
Risk distribution:
risk_level
Low        2613
High         46
Medium       29
Extreme       0

Top 10 highest hourly risk:
City           Timestamp  fire_probability risk_level
Baku 2026-05-07 06:00:00          0.431438       High
Baku 2026-05-08 06:00:00          0.430960       High
Baku 2026-05-09 06:00:00          0.430306       High
Baku 2026-05-06 06:00:00          0.429398       High
Baku 2026-05-10 06:00:00          0.426729       High
Baku 2026-05-07 05:00:00          0.406528       High
Baku 2026-05-08 05:00:00          0.406058       High
Baku 2026-05-09 05:00:00          0.405414       High
Baku 2026-05-06 05:00:00          0.404522       High
Baku

In [9]:
# ─── B6: Hourly dashboard JSON export ────────────────────────────────────
import shutil

h_out = h_risk.copy()
h_out["timestamp"]      = h_out["Timestamp"].dt.strftime("%Y-%m-%dT%H:%M")
h_out["region"]         = h_out["City"]
h_out["temperature"]    = h_out["Temperature_C"].round(1)
h_out["humidity"]       = h_out["Humidity_percent"].round(1)
h_out["wind"]           = h_out["Wind_Speed_kmh"].round(1)
h_out["solar"]          = h_out.get("Solar_Radiation_Wm2", pd.Series(dtype=float)).round(1)
h_out["probability"]    = h_out["fire_probability"].round(4)
h_out["risk_score"]     = (h_out["fire_probability"] * 100).round(1)
h_out["predicted_fire"] = h_out["fire_predicted"]
h_out["confidence"]     = h_out["fire_probability"].apply(
    lambda p: float(np.clip(0.55 + abs(p - 0.5) * 0.8, 0.55, 0.95)))
h_out["risk_color"]     = h_out["risk_level"].map({
    "Low": "#3FA773", "Medium": "#D8A31D", "High": "#D96C3B", "Extreme": "#B73333"})

hourly_public_cols = [
    "timestamp", "region", "temperature", "humidity", "wind", "solar",
    "probability", "risk_level", "risk_score", "predicted_fire",
    "confidence", "risk_color",
]
hourly_public_cols = [c for c in hourly_public_cols if c in h_out.columns]
h_public = h_out[hourly_public_cols].copy()

# Save to outputs/
(OUTPUTS / "hourly_forecast_168h.json").write_text(
    h_public.to_json(orient="records", indent=2), encoding="utf-8")

# Copy to dashboard & docs
for dest in [ROOT / "dashboard" / "data", ROOT / "docs" / "data"]:
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy2(OUTPUTS / "hourly_forecast_168h.json", dest / "hourly_forecast_168h.json")

print(f"Hourly forecast JSON saved ({len(h_public)} rows)")
print(f"  Timestamp range: {h_public['timestamp'].min()} → {h_public['timestamp'].max()}")
print(f"  Regions: {sorted(h_public['region'].unique())}")
print(f"  Includes fire risk: probability, risk_level, risk_score, predicted_fire")
print(f"  Copied to: dashboard/data/ + docs/data/")

# ── Update metrics.json with hourly info ─────────────────────────────────
metrics_path = OUTPUTS / "metrics.json"
if metrics_path.exists():
    with open(metrics_path) as f:
        metrics_data = json.load(f)
    metrics_data["hourly_model"] = {
        "model_name": h_manifest["model_name"],
        "optimal_threshold": float(H_THRESHOLD),
        "prediction_horizon_hours": 168,
        "n_features": len(H_FEATURE_COLS),
    }
    with open(metrics_path, "w") as f:
        json.dump(metrics_data, f, indent=2)
    for dest in [ROOT / "dashboard" / "data", ROOT / "docs" / "data"]:
        shutil.copy2(metrics_path, dest / "metrics.json")
    print("  Updated metrics.json with hourly model info")

print(f"\n→ Next: Dashboard toggle + 06_Climate_Report.ipynb")

Hourly forecast JSON saved (2688 rows)
  Timestamp range: 2026-05-03T20:00 → 2026-05-10T19:00
  Regions: ['Baku', 'Barda', 'Gabala', 'Ganja', 'Jalilabad', 'Khachmaz', 'Lankaran', 'Mingachevir', 'Nakhchivan', 'Quba', 'Shabran', 'Shaki', 'Shamakhi', 'Shirvan', 'Yevlakh', 'Zaqatala']
  Includes fire risk: probability, risk_level, risk_score, predicted_fire
  Copied to: dashboard/data/ + docs/data/
  Updated metrics.json with hourly model info

→ Next: Dashboard toggle + 06_Climate_Report.ipynb
